In [1]:
#Imports
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor

In [2]:
# Load + combine
df = pd.concat([
    pd.read_csv("../data/processed_22.csv"),
    pd.read_csv("../data/processed_23.csv"),
    pd.read_csv("../data/processed_24.csv"),
    pd.read_csv("../data/processed_25.csv")
], ignore_index=True)
df["date"] = pd.to_datetime(df["date"])

In [3]:
# Target: daily expected Lingulodinium polyedra per mL
y = df["Lpoly_expected_ml"].astype(float)
X = df.drop(columns=["date", "Lpoly_expected", "Lpoly_expected_ml"]).dropna(axis=1, how="all")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def eval_reg(name, model):
    model.fit(X_train, y_train)
    pred_test = model.predict(X_test)
    return {
        "Model": name,
        "Test_R2": float(r2_score(y_test, pred_test)),
        "Test_RMSE": rmse(y_test, pred_test),
    }


In [4]:
# Baselines
lin = Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())])
ridge = Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))])

rf = RandomForestRegressor(
    n_estimators=80,
    max_depth=10,
    random_state=42,
    n_jobs=1
)

perf = pd.DataFrame([
    eval_reg("Linear", lin),
    eval_reg("Ridge", ridge),
    eval_reg("RandomForest", rf),
]).sort_values("Test_R2", ascending=False)

print("Baseline regression performance (random split)")
print(perf)


Baseline regression performance (random split)
          Model   Test_R2  Test_RMSE
2  RandomForest  0.592879   7.747213
1         Ridge  0.350909   9.782203
0        Linear  0.242873  10.564970


In [5]:
# Feature importance

# 1) Ridge coefficients (standardized) = interpretable linear influence
ridge.fit(X_train, y_train)
coef = ridge.named_steps["model"].coef_
ridge_importance = pd.DataFrame({
    "feature": X.columns,
    "ridge_coef_abs": np.abs(coef),
    "ridge_coef_signed": coef
}).sort_values("ridge_coef_abs", ascending=False).reset_index(drop=True)

print("\nTop 20 Ridge coefficients (|coef|, standardized features)")
print(ridge_importance.head(20))

# 2) Random Forest impurity importance = nonlinear importance signal
rf.fit(X_train, y_train)
rf_importance = pd.DataFrame({
    "feature": X.columns,
    "rf_impurity_importance": rf.feature_importances_
}).sort_values("rf_impurity_importance", ascending=False).reset_index(drop=True)

print("\nTop 20 RandomForest impurity importances")
print(rf_importance.head(20))


Top 20 Ridge coefficients (|coef|, standardized features)
                                 feature  ridge_coef_abs  ridge_coef_signed
0                                 Ring05       10.779440         -10.779440
1                                  HOG34        8.320175           8.320175
2   summedConvexPerimeter_over_Perimeter        8.266895           8.266895
3                   texture_third_moment        8.086985          -8.086985
4             texture_average_gray_level        7.991270           7.991270
5                             ROI_per_ml        7.933867           7.933867
6                                  HOG17        7.269057           7.269057
7                                  HOG52        6.818484           6.818484
8              RotatedBoundingBox_xwidth        6.437335           6.437335
9           rotated_BoundingBox_solidity        6.349711          -6.349711
10                                 HOG53        6.297794          -6.297794
11                           

In [6]:
#Top 13 biological features only

# Explicitly define effort / metadata features to exclude
effort_features = ["ml_analyzed", "roiCount", "ROI_per_ml"]

top13_bio = [
    f for f in rf_importance["feature"].tolist()
    if f not in effort_features
][:13]

print("\nTop 13 biological features:")
print(top13_bio)

X_train_top13_bio = X_train[top13_bio]
X_test_top13_bio = X_test[top13_bio]

rf_top13_bio = RandomForestRegressor(
    n_estimators=80,
    max_depth=10,
    random_state=42,
    n_jobs=1
)

rf_top13_bio.fit(X_train_top13_bio, y_train)

#Results
y_pred_top13_bio = rf_top13_bio.predict(X_test_top13_bio)

r2_top13_bio = r2_score(y_test, y_pred_top13_bio)
rmse_top13_bio = np.sqrt(np.mean((y_test - y_pred_top13_bio) ** 2))

print("\nReduced top-13 BIOLOGICAL RF performance")
print(f"Test R2:   {r2_top13_bio:.3f}")
print(f"Test RMSE: {rmse_top13_bio:.3f}")

rf_top13_bio_importance = pd.DataFrame({
    "feature": top13_bio,
    "rf_impurity_importance": rf_top13_bio.feature_importances_
}).sort_values("rf_impurity_importance", ascending=False)

print("\nReduced RF feature importance (biological only)")
print(rf_top13_bio_importance)



Top 13 biological features:
['moment_invariant1', 'Ring10', 'moment_invariant2', 'moment_invariant3', 'Ring09', 'HOG79', 'HOG41', 'RWhalfpowerintegral', 'moment_invariant7', 'HOG40', 'HOG74', 'HOG18', 'texture_uniformity']

Reduced top-13 BIOLOGICAL RF performance
Test R2:   0.508
Test RMSE: 8.516

Reduced RF feature importance (biological only)
                feature  rf_impurity_importance
1                Ring10                0.221063
0     moment_invariant1                0.176806
4                Ring09                0.110490
8     moment_invariant7                0.095649
6                 HOG41                0.074859
2     moment_invariant2                0.070191
5                 HOG79                0.054397
3     moment_invariant3                0.050798
7   RWhalfpowerintegral                0.039099
12   texture_uniformity                0.034662
11                HOG18                0.030495
9                 HOG40                0.026790
10                HOG74    

In [8]:
df_all = pd.concat([
    pd.read_csv("../data/processed_22.csv"),
    pd.read_csv("../data/processed_23.csv"),
    pd.read_csv("../data/processed_24.csv"),
    pd.read_csv("../data/processed_25.csv")
], ignore_index=True)

df_all["date"] = pd.to_datetime(df_all["date"])

df_all.sort_values("Lpoly_expected_ml", ascending=False).head(10)[
    ["date", "Lpoly_expected_ml"]
]

,date,Lpoly_expected_ml
324,1970-01-01 00:00:00.020240518,172.389686
323,1970-01-01 00:00:00.020240517,131.889561
322,1970-01-01 00:00:00.020240516,85.849369
476,1970-01-01 00:00:00.020241105,75.519357
480,1970-01-01 00:00:00.020241109,65.297662
487,1970-01-01 00:00:00.020241122,61.108233
325,1970-01-01 00:00:00.020240519,53.397145
481,1970-01-01 00:00:00.020241110,48.266693
488,1970-01-01 00:00:00.020241123,46.417918
321,1970-01-01 00:00:00.020240515,44.784674
